In [1]:
!pip install pandas torch transformers tensorflow


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ===============================
# ENHANCED SENTIMENT ANALYSIS
# Matches the exact output format from the screenshot
# ===============================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datetime import datetime

# ===============================
# CONFIGURATION
# ===============================
NEWS_FILEPATH = "./STOCKNEPSE/NEPSEDATA/adblnews_full_content.csv"
OUTPUT_FILEPATH = "adbl_sentiment_results.csv"

# You can also process specific stock from multi-stock CSV
FILTER_BY_STOCK = None  # Set to "ADBL" or stock symbol to filter, None for all

# ===============================
# LOAD FINBERT
# ===============================
print("🤖 Loading FinBERT model...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
finbert_model.eval()
print("✅ FinBERT loaded\n")

# ===============================
# SENTIMENT ANALYSIS FUNCTION
# ===============================
def analyze_headline(headline):
    """
    Analyze a single headline and return detailed results
    
    Returns:
        dict with logits, prediction, and sentiment score
    """
    if not isinstance(headline, str) or headline.strip() == "" or headline.strip() == "0":
        return None
    
    headline = headline.strip()
    
    try:
        # Tokenize
        inputs = tokenizer(
            headline,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        
        # Get model output
        with torch.no_grad():
            outputs = finbert_model(**inputs)
            logits = outputs.logits[0].numpy()
            probs = F.softmax(outputs.logits, dim=1)[0].numpy()
            
            predicted_class = np.argmax(probs)
            confidence = probs[predicted_class]
        
        # Map to labels (FinBERT: 0=negative, 1=neutral, 2=positive)
        class_labels = {0: 'negative', 1: 'neutral', 2: 'positive'}
        prediction = class_labels[predicted_class]
        
        # Calculate sentiment score
        if predicted_class == 2:  # positive
            sentiment_score = float(confidence)
        elif predicted_class == 0:  # negative
            sentiment_score = float(-confidence)
        else:  # neutral
            sentiment_score = 0.0
        
        return {
            'logits': logits.tolist(),
            'prediction': prediction,
            'sentiment_score': sentiment_score,
            'confidence': float(confidence)
        }
        
    except Exception as e:
        print(f"Error analyzing headline: {e}")
        return None

# ===============================
# LOAD AND PROCESS DATA
# ===============================
print(f"📂 Loading data from: {NEWS_FILEPATH}")
news_df = pd.read_csv(NEWS_FILEPATH)
print(f"✅ Loaded {len(news_df)} rows\n")

# Initialize results list
results = []

print("⚙️  Processing headlines...")
print("="*80)

# Process each row
for idx, row in news_df.iterrows():
    # Extract date and headline
    # Adjust these column indices based on your CSV structure
    # Assuming: Column 0 = Date, Column 1 = Headline
    
    date = row.iloc[0] if len(row) > 0 else ""
    headline = row.iloc[1] if len(row) > 1 else ""
    
    # Analyze the headline
    analysis = analyze_headline(headline)
    
    if analysis is not None:
        # Format logits as string (matching your screenshot format)
        logit_str = f"[{analysis['logits'][0]:.4f}, {analysis['logits'][1]:.4f}, {analysis['logits'][2]:.4f}]"
        
        results.append({
            'Date': date,
            'Headline': headline,
            'Logit': logit_str,
            'Prediction': analysis['prediction'],
            'Sentiment_Score': analysis['sentiment_score']
        })
    
    # Progress indicator
    if (idx + 1) % 50 == 0:
        print(f"  Processed: {idx + 1}/{len(news_df)}")

print("="*80)
print(f"✅ Completed! Processed {len(results)} valid headlines\n")

# ===============================
# CREATE RESULTS DATAFRAME
# ===============================
results_df = pd.DataFrame(results)

# Sort by date (descending - most recent first)
try:
    results_df['Date'] = pd.to_datetime(results_df['Date'])
    results_df = results_df.sort_values('Date', ascending=False).reset_index(drop=True)
except:
    print("⚠️  Could not parse dates, keeping original order")

# ===============================
# DISPLAY SAMPLE
# ===============================
print("="*80)
print("📊 SAMPLE RESULTS (First 15 rows):")
print("="*80)
display_df = results_df.head(15).copy()

# Format for display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(display_df.to_string(index=True))
print("="*80 + "\n")

# ===============================
# SAVE TO CSV
# ===============================
results_df.to_csv(OUTPUT_FILEPATH, index=False)
print(f"💾 Results saved to: {OUTPUT_FILEPATH}")

# ===============================
# STATISTICS
# ===============================
print("\n" + "="*80)
print("📈 SENTIMENT ANALYSIS STATISTICS")
print("="*80)

# Prediction distribution
print("\n1. Prediction Distribution:")
print("-" * 40)
pred_counts = results_df['Prediction'].value_counts()
for pred in ['positive', 'neutral', 'negative']:
    count = pred_counts.get(pred, 0)
    pct = (count / len(results_df) * 100) if len(results_df) > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"  {pred.capitalize():<10}: {count:>4} ({pct:>5.1f}%) {bar}")

# Sentiment scores
print("\n2. Sentiment Score Statistics:")
print("-" * 40)
print(f"  Mean:      {results_df['Sentiment_Score'].mean():>8.4f}")
print(f"  Median:    {results_df['Sentiment_Score'].median():>8.4f}")
print(f"  Std Dev:   {results_df['Sentiment_Score'].std():>8.4f}")
print(f"  Min:       {results_df['Sentiment_Score'].min():>8.4f}")
print(f"  Max:       {results_df['Sentiment_Score'].max():>8.4f}")

# Zero sentiment count (neutral predictions)
zero_count = (results_df['Sentiment_Score'] == 0.0).sum()
print(f"  Zero (0.0): {zero_count:>4} ({zero_count/len(results_df)*100:.1f}%)")

# Overall sentiment
avg_score = results_df['Sentiment_Score'].mean()
print(f"\n3. Overall Market Sentiment:")
print("-" * 40)
if avg_score > 0.1:
    sentiment_emoji = "📈 BULLISH"
    color = "Positive"
elif avg_score < -0.1:
    sentiment_emoji = "📉 BEARISH"
    color = "Negative"
else:
    sentiment_emoji = "⏸️  NEUTRAL"
    color = "Neutral"

print(f"  {sentiment_emoji}")
print(f"  Score: {avg_score:.4f} ({color})")

# Top positive and negative headlines
print("\n4. Most Positive Headlines:")
print("-" * 40)
top_positive = results_df.nlargest(3, 'Sentiment_Score')[['Date', 'Headline', 'Sentiment_Score']]
for i, row in top_positive.iterrows():
    print(f"  [{row['Date']}] Score: {row['Sentiment_Score']:.4f}")
    print(f"    {row['Headline'][:70]}...")

print("\n5. Most Negative Headlines:")
print("-" * 40)
top_negative = results_df.nsmallest(3, 'Sentiment_Score')[['Date', 'Headline', 'Sentiment_Score']]
for i, row in top_negative.iterrows():
    print(f"  [{row['Date']}] Score: {row['Sentiment_Score']:.4f}")
    print(f"    {row['Headline'][:70]}...")

print("\n" + "="*80)
print(f"✅ Analysis complete! Open '{OUTPUT_FILEPATH}' in Excel or Jupyter to view.")
print("="*80)

# ===============================
# OPTIONAL: SAVE EXCEL VERSION
# ===============================
try:
    excel_path = OUTPUT_FILEPATH.replace('.csv', '.xlsx')
    results_df.to_excel(excel_path, index=False)
    print(f"\n📊 Excel version saved to: {excel_path}")
except:
    print("\n⚠️  Could not save Excel version (openpyxl not installed)")
    print("   Install with: pip install openpyxl")

🤖 Loading FinBERT model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FinBERT loaded

📂 Loading data from: ./STOCKNEPSE/NEPSEDATA/adblnews_full_content.csv
✅ Loaded 496 rows

⚙️  Processing headlines...
  Processed: 50/496
  Processed: 100/496
  Processed: 150/496
  Processed: 200/496
  Processed: 250/496
  Processed: 300/496
  Processed: 350/496
  Processed: 400/496
  Processed: 450/496
✅ Completed! Processed 496 valid headlines

📊 SAMPLE RESULTS (First 15 rows):
         Date                                                                                                                               Headline                       Logit Prediction  Sentiment_Score
0  2026-02-17                                                                                     Bonus Shares of ADBL, ALBSL, SHIVM listed in NEPSE  [-0.8137, -1.4031, 2.6527]   positive         0.953698
1  2026-02-12                                Falgun Interest Rate Update: Commercial Banks Revise Fixed Deposit Rates; Majority Keep Rates Unchanged  [-0.3563, -1.0445, 1.6315]   positive  